In [1]:
!apt-get update
!apt-get -y install git python3-pip gcc-10 g++-10 zlib1g zlib1g-dev
!pip install --upgrade meson ninja

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [75.3 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,919 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu n

In [2]:
!git clone -b release/0.32 https://github.com/LeelaChessZero/lc0.git

Cloning into 'lc0'...
remote: Enumerating objects: 14093, done.
remote: Counting objects: 100% (1816/1816), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 14093 (delta 1671), reused 1583 (delta 1583), pack-reused 12277 (from 3)
Receiving objects: 100% (14093/14093), 40.28 MiB | 18.42 MiB/s, done.
Resolving deltas: 100% (10327/10327), done.


In [3]:
%cd /kaggle/working/lc0
!CC=gcc-10 CXX=g++-10 meson setup build/release --buildtype=release -Dbackend_max_links=1
!ninja -C build/release lc0 -j 1

/kaggle/working/lc0
The Meson build system
Version: 1.12.1
Source dir: /kaggle/working/lc0
Build dir: /kaggle/working/lc0/build/release
Build type: native build
DEPRECATION: WrapDB v1 is deprecated, updated using `meson wrap update zlib`
Project name: lc0
Project version: undefined
C++ compiler for the host machine: g++-10 (gcc 10.5.0 "g++-10 (Ubuntu 10.5.0-4ubuntu2.1) 10.5.0")
C++ linker for the host machine: g++-10 ld.bfd 2.42
Host machine cpu family: x86_64
Host machine cpu: x86_64
Compiler for C++ supports arguments -march=native: YES 
Program scripts/compile_proto.py found: YES (/kaggle/working/lc0/scripts/compile_proto.py)
Program git found: YES (/usr/bin/git)
Message: updating git submodule libs/lczero-common
Configuring build_id.h using configuration
Downloading: 100%|██████████████████████████████| 2.14M/2.14M 16.5MB/s eta 00:00

Executing subproject abseil-cpp for machine: host 

abseil-cpp| Project name: abseil-cpp
abseil-cpp| Project version: 20240722.0
abseil-cpp| C++ comp

In [4]:
%cd /kaggle/working/lc0/build/release
!rm -f weights.pb.gz
!wget -q --show-progress https://storage.lczero.org/files/networks-contrib/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz -O weights.pb.gz

/kaggle/working/lc0/build/release
weights.pb.gz       100%[===================>] 364.92M   304MB/s    in 1.2s    


In [5]:
!./lc0 --help

       _
|   _ | |
|_ |_ |_| v0.32.1 built Oct 19 2026
Search algorithm: classic
Usage: ./lc0 [<mode>] [flags...]

Available modes. A help for a mode: ./lc0 <mode> --help
  selfplay   Play games with itself
  benchmark  Quick benchmark
  bench      Very quick benchmark
  backendbench Quick benchmark of backend only
  leela2onnx Convert Leela network to ONNX.
  onnx2leela Convert ONNX network to Leela net.
  describenet Shows details about the Leela network.
  dag-preview Use "dag-preview" search
  policyhead Use "policyhead" search
  valuehead  Use "valuehead" search
  classic    Use "classic" search

Allowed command line flags for current mode:
  -h,  --help  Show help and exit.

       --show-hidden
               Show hidden options. Use with --help.

  -l,  --logfile=STRING
               Write log to that file. Special value <stderr> to output the log to the console.
               [UCI: LogFile]

  -c,  --config=STRING
               Path to a configuration file. The format of th

In [6]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 46.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147879 sha256=b1c30c7baac849042e4ad5b22fa9d7d24ff27c80efe74d599bc33a1bfb41d2be
  Stored in directory: /root/.cache/pip/wheels/74/65/99/8c3a88ba852c1ed8d34daed144b8cb4c654d638677483d19ff
Successfully built chess


In [7]:
import chess
import chess.engine
import csv
import math

# --- KAGGLE Lc0 CONFIGURATION ---
LC0_PATH = "/kaggle/working/lc0/build/release/lc0" 
WEIGHTS_PATH = "/kaggle/working/lc0/build/release/weights.pb.gz"
INPUT_FILE = "/kaggle/input/datasets/sslm11/1000000chesspositions/optimal_positions.txt"
OUTPUT_FILE = "/kaggle/working/lc0_erel_qvalues_nodes.csv"

# 1. Define the Node Targets: 1 node + 10^(n/4 + 1) for n in 0..20
NODE_TARGETS = [1] + [int(10**((n / 4) + 1)) for n in range(21)]

# 2. Define the normalized epsilon baskets for a 0 to 1 relative scale
EPSILONS = [round(x * 0.05, 2) for x in range(21)]

def run_lc0_analysis():
    print("Spinning up Leela Chess Zero (Lc0)...", flush=True)
    try:
        engine = chess.engine.SimpleEngine.popen_uci(LC0_PATH)
        engine.configure({
            "ScoreType": "Q", 
            "WeightsFile": WEIGHTS_PATH,
            "Backend": "demux", 
            "BackendOptions": "backend=cuda-fp16,(gpu=0),(gpu=1)",
            "MinibatchSize": 256,
            "Threads": 2 
        })
    except FileNotFoundError:
        print(f"Error: Lc0 not found at {LC0_PATH}.")
        return

    with open(INPUT_FILE, "r") as f:
        # We only need 100 positions for the actual data extraction
        fens_to_evaluate = [line.strip() for line in f if line.strip()][:100]
    
    print(f"Loaded {len(fens_to_evaluate)} positions. Commencing evaluation...", flush=True)

    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        header = ["FEN", "Target_Nodes"] + [f"Rel_Error_<={e:.2f}" for e in EPSILONS]
        writer.writerow(header)

        for i, fen in enumerate(fens_to_evaluate, start=1):
            board = chess.Board(fen)
            legal_moves = list(board.legal_moves)
            num_moves = len(legal_moves)
            
            if num_moves == 0:
                continue
                
            print(f"  Analyzing Position {i}/100...", flush=True)
            
            # Loop through the mathematically defined node targets
            for target_nodes in NODE_TARGETS:
                analysis = engine.analyse(board, chess.engine.Limit(nodes=target_nodes), multipv=num_moves)
                
                q_values = {}
                for info_dict in analysis:
                    pv = info_dict.get("pv")
                    if not pv:
                        continue
                        
                    move = pv[0]
                    score_obj = info_dict.get("score")
                    
                    if score_obj is not None:
                        # Convert ScoreType=Q back to [-1.0, 1.0] probablity
                        val = score_obj.pov(board.turn).score(mate_score=100)
                        q_values[move] = val / 100.0
                        
                if not q_values:
                    continue

                max_q = max(q_values.values())
                min_q = min(q_values.values())
                q_range = max_q - min_q
                
                epsilon_counts = {e: 0 for e in EPSILONS}
                
                # e_rel = Delta_i / Delta_max calculation
                for move, q in q_values.items():
                    loss = max_q - q
                    rel_error = (loss / q_range) if q_range > 0 else 0.0
                    rel_error = min(1.0, max(0.0, rel_error))
                    
                    for e in EPSILONS:
                        if rel_error <= e:
                            epsilon_counts[e] += 1
                            
                row = [fen, target_nodes]
                for e in EPSILONS:
                    rel_freq = epsilon_counts[e] / num_moves
                    row.append(rel_freq)
                    
                writer.writerow(row)
            
            csv_file.flush() 

    engine.quit()
    print("\nExtraction complete! Data saved under the relative error normalization framework.", flush=True)

if __name__ == "__main__":
    run_lc0_analysis()

Spinning up Leela Chess Zero (Lc0)...


<UciProtocol (pid=2813)>: stderr >>        _
<UciProtocol (pid=2813)>: stderr >> |   _ | |
<UciProtocol (pid=2813)>: stderr >> |_ |_ |_| v0.32.1 built Oct 19 2026
<UciProtocol (pid=2813)>: stderr >> Search algorithm: classic


Loaded 100 positions. Commencing evaluation...
  Analyzing Position 1/100...


<UciProtocol (pid=2813)>: stderr >> Loading weights file from: /kaggle/working/lc0/build/release/weights.pb.gz
<UciProtocol (pid=2813)>: stderr >> Weights file has multihead format, updating format flag
<UciProtocol (pid=2813)>: stderr >> Creating backend [cuda-fp16]...
<UciProtocol (pid=2813)>: stderr >> CUDA Runtime version: 12.8.0
<UciProtocol (pid=2813)>: stderr >> Latest version of CUDA supported by the driver: 13.0.0
<UciProtocol (pid=2813)>: stderr >> GPU: Tesla T4
<UciProtocol (pid=2813)>: stderr >> GPU memory: 14.5622 Gb
<UciProtocol (pid=2813)>: stderr >> GPU clock frequency: 1590 MHz
<UciProtocol (pid=2813)>: stderr >> GPU compute capability: 7.5
<UciProtocol (pid=2813)>: stderr >> L2 cache capacity: 4194304
<UciProtocol (pid=2813)>: stderr >> Creating backend [cuda-fp16]...
<UciProtocol (pid=2813)>: stderr >> CUDA Runtime version: 12.8.0
<UciProtocol (pid=2813)>: stderr >> Latest version of CUDA supported by the driver: 13.0.0
<UciProtocol (pid=2813)>: stderr >> GPU: Tesla 

  Analyzing Position 2/100...
  Analyzing Position 3/100...
  Analyzing Position 4/100...
  Analyzing Position 5/100...
  Analyzing Position 6/100...
  Analyzing Position 7/100...
  Analyzing Position 8/100...
  Analyzing Position 9/100...
  Analyzing Position 10/100...
  Analyzing Position 11/100...
  Analyzing Position 12/100...
  Analyzing Position 13/100...
  Analyzing Position 14/100...
  Analyzing Position 15/100...
  Analyzing Position 16/100...
  Analyzing Position 17/100...
  Analyzing Position 18/100...
  Analyzing Position 19/100...
  Analyzing Position 20/100...
  Analyzing Position 21/100...
  Analyzing Position 22/100...
  Analyzing Position 23/100...
  Analyzing Position 24/100...
  Analyzing Position 25/100...
  Analyzing Position 26/100...
  Analyzing Position 27/100...
  Analyzing Position 28/100...
  Analyzing Position 29/100...
  Analyzing Position 30/100...
  Analyzing Position 31/100...
  Analyzing Position 32/100...
  Analyzing Position 33/100...
  Analyzing Posi